# Phase 3: Cross-Encoder Reranking

This notebook retrieves a larger candidate set with the existing hybrid retriever and reranks every query-document pair with `cross-encoder/ms-marco-MiniLM-L-6-v2`. All inference runs locally in the Colab runtime.

## Environment

Replace `REPOSITORY_URL` with your repository URL before running this notebook in Colab.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/YOUR_GITHUB_USERNAME/adaptive-rag.git'
PROJECT_DIR = Path('/content/adaptive-rag')

if not (PROJECT_DIR / 'src').exists():
    if 'YOUR_GITHUB_USERNAME' in REPOSITORY_URL:
        raise RuntimeError('Set REPOSITORY_URL to your GitHub repository URL, then run this cell again.')
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(f'Working directory: {Path.cwd()}')

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'unavailable')

## Initialize Phase 1–3 retrieval components

FAISS and BM25 use the same controlled corpus. RRF creates the candidate ranking, then the cross-encoder evaluates each candidate jointly with the query.

In [ ]:
from src.rag import (
    BM25Retriever,
    CrossEncoderReranker,
    FAISSRetriever,
    HybridRetriever,
    RAGConfig,
    load_documents,
)

config = RAGConfig()
documents = load_documents(Path('data/phase1_corpus.json'))
index_dir = Path('data/phase1_faiss_index')

dense_retriever = FAISSRetriever(
    embedding_model_name=config.embedding_model_name,
    device=config.device,
    batch_size=config.embedding_batch_size,
)
if (index_dir / 'documents.faiss').exists():
    dense_retriever.load(index_dir)
    print(f'Loaded FAISS index with {len(dense_retriever.documents)} documents.')
else:
    dense_retriever.build(documents)
    dense_retriever.save(index_dir)
    print(f'Built FAISS index with {len(documents)} documents.')

bm25_retriever = BM25Retriever(documents)
hybrid_retriever = HybridRetriever(dense_retriever, bm25_retriever)
reranker = CrossEncoderReranker(
    model_name='cross-encoder/ms-marco-MiniLM-L-6-v2',
    device=config.device,
)
print('Reranker model:', reranker.model_name)
print('Reranker device:', reranker.device)

## Compare rankings before and after reranking

The reranker scores are relevance logits used for ordering; they should not be interpreted as calibrated probabilities.

In [ ]:
def compare_ranking(query: str, candidate_k: int = 10, final_k: int = 5):
    candidates = hybrid_retriever.retrieve(query, top_k=candidate_k)
    reranked = reranker.rerank(query, candidates, top_k=final_k)

    print('\n' + '=' * 100)
    print('Query:', query)
    print(f'\nBefore reranking — top {len(candidates)} hybrid candidates')
    for rank, result in enumerate(candidates, start=1):
        print(f'  Rank {rank}: {result.title} (id={result.id}, RRF={result.score:.6f}, dense_rank={result.dense_rank}, bm25_rank={result.bm25_rank})')

    print(f'\nAfter reranking — top {len(reranked)}')
    for rank, result in enumerate(reranked, start=1):
        print(f'  Rank {rank}: {result.title} (id={result.id}, reranker={result.reranker_score:.6f}, original_RRF={result.rrf_score:.6f}, dense_rank={result.dense_rank}, bm25_rank={result.bm25_rank})')
        print(f'    {result.text}')
    return candidates, reranked

In [ ]:
example_queries = [
    'IPv4 uses 32-bit addresses',
    'Which scientist won major awards in two different scientific fields?',
    'What object in space is chiefly responsible for tides in Earth’s oceans?',
]

for example_query in example_queries:
    compare_ranking(example_query, candidate_k=10, final_k=5)